Loading PDF Files

In [4]:
from langchain_community.document_loaders import(
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)

e:\Study\rag-repo-public\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
## method 1- PyPDF Loader
print("PyPDFLoader")

try:
    pypdf_loader=PyMuPDFLoader("data/pdf/sample.pdf")
    pypdf_doc=pypdf_loader.load()
    print("Document: ",pypdf_doc)
    print(f"Loaded {len(pypdf_doc)} documents")
    print(f"---------------------------------")
    print(f"content preview: {pypdf_doc[0].page_content[:200]}")
    print(f"---------------------------------")
    print(f"content preview: {pypdf_doc[0].metadata}")
except Exception as e:
    print(f"Error: {e}")

PyPDFLoader
Document:  [Document(metadata={'producer': 'Mac OS X 10.5.4 Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20080701052447Z00'00'", 'source': 'data/pdf/sample.pdf', 'file_path': 'data/pdf/sample.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'sample', 'author': 'Philip Hutchison', 'subject': '', 'keywords': '', 'moddate': "D:20080701052447Z00'00'", 'trapped': '', 'modDate': "D:20080701052447Z00'00'", 'creationDate': "D:20080701052447Z00'00'", 'page': 0}, page_content='Sample PDF\nThis is a simple PDF ﬁle. Fun fun fun.\nLorem ipsum dolor sit amet, consectetuer adipiscing elit. Phasellus facilisis odio sed mi. \nCurabitur suscipit. Nullam vel nisi. Etiam semper ipsum ut lectus. Proin aliquam, erat eget \npharetra commodo, eros mi condimentum quam, sed commodo justo quam ut velit. \nInteger a erat. Cras laoreet ligula cursus enim. Aenean scelerisque velit et tellus. \nVestibulum dictum aliquet sem. Nulla facilisi. Vestibulum accumsan ante vitae elit. Nulla \n

In [6]:
## method 2- PyMuPDF Loader
print("PyMuPDFLoader")

try:
    pymupdf_loader=PyMuPDFLoader("data/pdf/sample.pdf")
    pymupdf_doc=pymupdf_loader.load()
    print("Document: ",pymupdf_doc)
    print(f"Loaded {len(pymupdf_doc)} documents")
    print(f"---------------------------------")
    print(f"content preview: {pymupdf_doc[0].page_content[:200]}")
    print(f"---------------------------------")
    print(f"content preview: {pymupdf_doc[0].metadata}")
except Exception as e:
    print(f"Error: {e}")

PyMuPDFLoader
Document:  [Document(metadata={'producer': 'Mac OS X 10.5.4 Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20080701052447Z00'00'", 'source': 'data/pdf/sample.pdf', 'file_path': 'data/pdf/sample.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'sample', 'author': 'Philip Hutchison', 'subject': '', 'keywords': '', 'moddate': "D:20080701052447Z00'00'", 'trapped': '', 'modDate': "D:20080701052447Z00'00'", 'creationDate': "D:20080701052447Z00'00'", 'page': 0}, page_content='Sample PDF\nThis is a simple PDF ﬁle. Fun fun fun.\nLorem ipsum dolor sit amet, consectetuer adipiscing elit. Phasellus facilisis odio sed mi. \nCurabitur suscipit. Nullam vel nisi. Etiam semper ipsum ut lectus. Proin aliquam, erat eget \npharetra commodo, eros mi condimentum quam, sed commodo justo quam ut velit. \nInteger a erat. Cras laoreet ligula cursus enim. Aenean scelerisque velit et tellus. \nVestibulum dictum aliquet sem. Nulla facilisi. Vestibulum accumsan ante vitae elit. Nulla 

In [7]:
## PDF Cleaning

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List,Dict,Any


class SmartPDFProcessor:
    def __init__(self,chunk_size=10,chunk_overlap=5):
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap
        self.text_splitter=RecursiveCharacterTextSplitter(
            separators=[" "],
            chunk_overlap=chunk_overlap,
            chunk_size=chunk_size
        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with mart chunking and metadata enhancement"""

        #Load PDF
        loader=PyMuPDFLoader(pdf_path)
        pages=loader.load()

        ## process each page
        processed_chunks=[]

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text=self._clean_text(page.page_content)

            ## skip empty pages
            if len(cleaned_text.strip()) < 50:
                continue


            ## create chunks with enhanced metadata
            chunks=self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks
    
    def _clean_text(self,text:str) -> str:
        ## remove extra space
        text = " ".join(text.split())

        ## fix common PDF extraction issues
        text=text.replace("fi","fi")
        text=text.replace("fl","fl")

        return text
    
preprocessor=SmartPDFProcessor()
try:
    smart_chunks=preprocessor.process_pdf("data/pdf/sample.pdf")
    print(f"Lenght: {len(smart_chunks)}")
    print(f"Content: {smart_chunks}")
    print(f"Metadata: {smart_chunks[0].metadata}")
    print(f"Page Content: {smart_chunks[0].page_content}")
    print("---------------------------------------------")

    if smart_chunks:
        for key,value in smart_chunks[0].metadata.items():
            print(f"{key} : {value}")
except Exception as e:
    print(f"Error: {e}")



Lenght: 363
Content: [Document(metadata={'producer': 'Mac OS X 10.5.4 Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20080701052447Z00'00'", 'source': 'data/pdf/sample.pdf', 'file_path': 'data/pdf/sample.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'sample', 'author': 'Philip Hutchison', 'subject': '', 'keywords': '', 'moddate': "D:20080701052447Z00'00'", 'trapped': '', 'modDate': "D:20080701052447Z00'00'", 'creationDate': "D:20080701052447Z00'00'", 'page': 1, 'chunk_method': 'smart_pdf_processor', 'char_count': 2847}, page_content='Sample PDF'), Document(metadata={'producer': 'Mac OS X 10.5.4 Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20080701052447Z00'00'", 'source': 'data/pdf/sample.pdf', 'file_path': 'data/pdf/sample.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'sample', 'author': 'Philip Hutchison', 'subject': '', 'keywords': '', 'moddate': "D:20080701052447Z00'00'", 'trapped': '', 'modDate': "D:20080701052447Z00'00'", 'creationDate': 